In [1]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import torch.nn as nn
import torch.optim as optim

from helper_modules import phase1_preprocess_data, phase2_preprocess_data

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


### Read in the (test/submission) data

In [2]:
df_test = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_test.csv'))
df_test = df_test.sort_values(by=['Site', 'Timestamp_Local'])
# df_test = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_test_partial.csv'))
df_test['Demand_Response_Capacity_kW'] = np.nan  # Add the target column with NaN values so that it can be processed easily later
print(df_test.shape)
display(df_test.head())

(105120, 7)


,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Demand_Response_Capacity_kW
0,siteD,2020-01-01 00:00:00,24.70,0,4.8,0.0,NaN
1,siteD,2020-01-01 00:15:00,24.61,0,4.8,0.0,NaN
2,siteD,2020-01-01 00:30:00,24.53,0,4.8,0.0,NaN
3,siteD,2020-01-01 00:45:00,24.45,0,4.8,0.0,NaN
4,siteD,2020-01-01 01:00:00,24.36,0,4.8,0.0,NaN


In [3]:
# Separate df_test into two parts: null and not null for 'Demand_Response_Flag'
df_test_null = df_test[df_test['Demand_Response_Flag'].isnull()].copy(deep=True)
df_test_notnull = df_test[~df_test['Demand_Response_Flag'].isnull()].copy(deep=True)

### Stage 1 Predictions -- Classifying Demand_Response_Flag
- Only do this for df_test_null

In [4]:
# perform some preprocessing for phase 1
df = phase1_preprocess_data(df_test_null)
df.drop(columns=['Demand_Response_Flag'], inplace=True)

In [5]:
# Load the trained model from file
from net_architecture import Net    # Define the neural network architecture: Just need to load architecture defined in net_architecture.py
input_dim = df.shape[1]             # Number of features
num_classes = 3                     # Number of classes in target variable
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('./models/phase1_nn_model.pth'))

# load the scaler, which should be ColumnTransformer type
# scaler is a combination of PowerTransformer (Yeo-Johnson transformation) & StandardScaler
scaler = joblib.load('./models/phase1_scaler.pkl')
print(scaler)

ColumnTransformer(transformers=[('bin_pass', 'passthrough',
                                 ['is_daylight', 'Is_Weekend', 'Is_Summer',
                                  'Is_Winter', 'Is_Afternoon', 'Is_Evening']),
                                ('yj_std',
                                 Pipeline(steps=[('yeojohnson',
                                                  PowerTransformer(standardize=False)),
                                                 ('std', StandardScaler())]),
                                 ['Dry_Bulb_Temperature_C',
                                  'Global_Horizontal_Radiation_W/m2',
                                  'Building_Power_kW', 'Hour', 'Day', 'DOW',
                                  'Month', 'Weekday', 'Minute', 'Hour_sin',
                                  'Hour_cos', 'DOW_sin', 'DOW_cos', 'HDD18',
                                  'CDD22', 'TempC2', 'rad_sqrt', 'rad_log1p',
                                  'TempC_x_daylight', 'CDD22_x_daylight',


In [6]:
# Convert DataFrame to torch tensor
X = scaler.transform(df)
X = torch.tensor(X, dtype=torch.float32)

# Set model to evaluation mode
model_loaded.eval()
with torch.no_grad():
    outputs = model_loaded(X)
    predictions = torch.argmax(outputs, dim=1)

# convert 2 in predictions to -1
predictions = np.where(predictions == 2, -1, predictions)

# Calculate the distribution of predictions classes
unique, counts = np.unique(predictions, return_counts=True)
distribution_pred = dict(zip(unique, counts))
print(distribution_pred)

{np.int64(-1): np.int64(10212), np.int64(0): np.int64(51464), np.int64(1): np.int64(8404)}


In [7]:
df_test_null.head()

,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Demand_Response_Capacity_kW
35040,siteE,2020-01-01 00:00:00,15.50,0,4.8,NaN,NaN
35041,siteE,2020-01-01 00:15:00,15.17,0,4.8,NaN,NaN
35042,siteE,2020-01-01 00:30:00,14.85,0,4.8,NaN,NaN
35043,siteE,2020-01-01 00:45:00,14.52,0,4.8,NaN,NaN
35044,siteE,2020-01-01 01:00:00,14.20,0,4.8,NaN,NaN


In [8]:
# Prepare data for Stage 2 predictions:
# make predictions for df_test_null by using the predictions from Stage 1
df_stage2 = df_test_null.drop(columns=['Demand_Response_Flag'])
df_stage2['Demand_Response_Flag'] = predictions
# then combine df_test_null and df_test_notnull back to df_stage2
df_stage2 = pd.concat([df_stage2, df_test_notnull], axis=0)
df_stage2 = df_stage2.sort_values(by=['Site', 'Timestamp_Local'])
df_stage2.reset_index(drop=True, inplace=True)

### Stage 2 Predictions - Demand_Response_Capacity_kW

In [9]:
df = df_stage2.copy(deep=True)

# For rows with Demand_Response_Flag == 0, set Demand_Response_Capacity_kW = 0.0
df_zero = df[df['Demand_Response_Flag'] == 0].copy(deep=True)
df_zero['Demand_Response_Capacity_kW'] = 0.0
df_zero = df_zero[['Site','Timestamp_Local','Demand_Response_Flag','Demand_Response_Capacity_kW']].copy(deep=True)

# For rows with Demand_Response_Flag == 1, we will predict Demand_Response_Capacity_kW
df_pos = df_stage2.copy(deep=True)
df_pos = df_pos[df_pos['Demand_Response_Flag'] == 1].copy(deep=True)
df = phase2_preprocess_data(df_pos)
df.drop(columns=['Demand_Response_Capacity_kW','Demand_Response_Flag'], inplace=True)
X = df.values
reg_pos = joblib.load('./models/phase2_xgb_reg_pos_model.pkl')
y_pred = reg_pos.predict(X)
df_pos = df_pos[['Site','Timestamp_Local','Demand_Response_Flag']].copy(deep=True)
df_pos['Demand_Response_Capacity_kW'] = y_pred

# For rows with Demand_Response_Flag == -1, we will predict Demand_Response_Capacity_kW
df_neg = df_stage2.copy(deep=True)
df_neg = df_neg[df_neg['Demand_Response_Flag'] == -1].copy(deep=True)
df = phase2_preprocess_data(df_neg)
df.drop(columns=['Demand_Response_Capacity_kW','Demand_Response_Flag'], inplace=True)
X = df.values
reg_neg = joblib.load('./models/phase2_xgb_reg_neg_model.pkl')
y_pred = reg_neg.predict(X)
df_neg = df_neg[['Site','Timestamp_Local','Demand_Response_Flag']].copy(deep=True)
df_neg['Demand_Response_Capacity_kW'] = y_pred

ValueError: Feature shape mismatch, expected: 28, got 12

### Submission

In [ ]:
# Combine all results
df_submission = pd.concat([df_zero, df_pos, df_neg], axis=0)
df_submission = df_submission.sort_values(by=['Site', 'Timestamp_Local'])
df_submission.reset_index(drop=True, inplace=True)

df_submission.to_csv('./submissions/submission.csv', index=False)
print("Submission file created: submission.csv in ./submissions folder")

In [ ]:
display(df_submission.head())
print('submission file shape:', df_submission.shape)
df_submission['Demand_Response_Flag'].value_counts()

In [ ]:
df_submission['Demand_Response_Capacity_kW'].hist(bins=100)

In [ ]:
regression_values = df_submission['Demand_Response_Capacity_kW'].values
print(np.min(regression_values), np.max(regression_values), np.mean(regression_values), np.median(regression_values))
df_submission[
    df_submission['Demand_Response_Capacity_kW'] != 0
]['Demand_Response_Capacity_kW'].hist(bins=100)

In [ ]:
X.head()